# Getting Started with dscompanion

The fastest path to a working `dscompanion` pipeline: install, fetch a real dataset,
run the smallest config that works, and look at what came back. No deep explanation
here — that's what [`02_classification_bank_marketing.ipynb`](02_classification_bank_marketing.ipynb)
is for. See [`README.md`](README.md) for the full notebook index.

In [1]:
!pip install -q dscompanion[notebooks]

zsh:1: no matches found: dscompanion[notebooks]


## The dataset

We'll use the UCI **Bank Marketing** dataset — records from a Portuguese bank's phone
marketing campaign for term deposits. The target column is `y`: did the client
subscribe (`"yes"`/`"no"`)? That makes this a binary classification problem.

dscompanion's classification models expect a **numeric** target (`0`/`1`), not a
categorical string label — so we map `"yes"`/`"no"` to `1`/`0` before saving.

In [2]:
from pathlib import Path
from ucimlrepo import fetch_ucirepo

cache_dir = Path(".data_cache")
cache_dir.mkdir(exist_ok=True)
data_path = cache_dir / "bank_marketing.parquet"

if not data_path.exists():
    ds = fetch_ucirepo(id=222)
    df = ds.data.features.copy()
    df["y"] = ds.data.targets["y"].map({"yes": 1, "no": 0})
    df.to_parquet(data_path, index=False)
else:
    import pandas as pd
    df = pd.read_parquet(data_path)

print(f"Saved {data_path} — {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Saved .data_cache/bank_marketing.parquet — 45211 rows, 17 columns


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,0
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,0
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,0
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,0


## The config

Every dscompanion run starts with a `PipelineConfig` — one object describing the
whole pipeline. Here's the smallest one that works: a data source, a target column,
and a task type. Everything else (split strategy, feature processing, algorithm choice,
evaluation) falls back to sensible defaults.

In [3]:
from dscompanion.pipeline import PipelineConfig, PipelineRunner

cfg = PipelineConfig(
    name="getting_started",
    data={"path": str(data_path), "format": "parquet", "target": "y"},
    model={"task": "classification"},
)
result = PipelineRunner(cfg).run()

2026-09-13 11:19:39  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:39  INFO      dscompanion.pipeline.runner  PipelineRunner  |  getting_started  v1.0


2026-09-13 11:19:39  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: classification  |  Algorithm: xgboost


2026-09-13 11:19:39  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:39  INFO      dscompanion.pipeline.runner  Run directory: reports/20260913_111939


...

2026-09-13 11:19:43  INFO      dscompanion.tracking.run_context  Run finished — 20260913


2026-09-13 11:19:43  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:19:43  INFO      dscompanion.pipeline.runner  Pipeline complete — 3.9s  |  Run: 20260913_111939


2026-09-13 11:19:43  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:19:43  INFO      dscompanion.pipeline.runner  ============================================================


## What just happened

`PipelineRunner.run()` did everything in one call: loaded the data, split it into
train/test, ran EDA, processed features (imputation, encoding, scaling), selected
features, trained a model, and evaluated it. The returned `result` carries every
artifact from that run.

In [4]:
result.metrics

,split,metric,value
0,train,f1,0.619107
1,train,precision,0.756061
2,train,recall,0.524160
3,train,roc_auc,0.950745
4,train,gini,0.901489
5,train,ks_statistic,0.766160
6,train,log_loss,0.178049
7,test,f1,0.501399
8,test,precision,0.614540
9,test,recall,0.423440


`result.metrics` is a tidy (long-format) table: one row per `(split, metric)` pair,
with the value in the `value` column. That shape stays consistent across every task
dscompanion supports — classification, regression, and clustering all return metrics
this way, just with different metric names.

Below: the fitted model itself, and where this run's artifacts (logs, reports, the
trained model file) were written on disk.

In [5]:
print(result.model)
print(result.run_dir)

reports/20260913_111939


## Next

[`02_classification_bank_marketing.ipynb`](02_classification_bank_marketing.ipynb) goes
much deeper on this same dataset: EDA reports, why the split strategy matters, comparing
multiple algorithms via the leaderboard, hyperparameter tuning, explainability (SHAP and
permutation importance), and exporting a governance-ready model card.